# ChrisGoesGolfing — Parameter Golf Experiment Analysis

Analysis of autonomous hyperparameter tuning results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 6 columns)
df = pd.read_csv("results.tsv", sep="\t")
df["val_bpb"] = pd.to_numeric(df["val_bpb"], errors="coerce")
df["val_bpb_quant"] = pd.to_numeric(df["val_bpb_quant"], errors="coerce")
df["artifact_bytes"] = pd.to_numeric(df["artifact_bytes"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    print(f"  #{i:3d}  bpb_q={row['val_bpb_quant']:.6f}  artifact={row['artifact_bytes']:.0f}  {row['description']}")

## Val BPB (Quantized) Over Time

Track how the best val_bpb_quant evolves. The running minimum shows the frontier.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12), gridspec_kw={'height_ratios': [3, 1]})

# Filter out crashes
valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
baseline_bpb = valid.loc[0, "val_bpb_quant"]

# BPB plot
below = valid[valid["val_bpb_quant"] <= baseline_bpb + 0.001]
disc = below[below["status"] == "DISCARD"]
ax1.scatter(disc.index, disc["val_bpb_quant"], c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

kept_v = below[below["status"] == "KEEP"]
ax1.scatter(kept_v.index, kept_v["val_bpb_quant"], c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_bpb = valid.loc[kept_mask, "val_bpb_quant"]
running_min = kept_bpb.cummin()
best = running_min.iloc[-1]
ax1.step(kept_idx, running_min, where="post", color="#27ae60", linewidth=2, alpha=0.7, zorder=3, label="Running best")

for idx, bpb in zip(kept_idx, kept_bpb):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 40: desc = desc[:37] + "..."
    ax1.annotate(desc, (idx, bpb), textcoords="offset points", xytext=(6, 6), fontsize=7.5, color="#1a7a3a", alpha=0.9, rotation=30, ha="left", va="bottom")

ax1.set_ylabel("val_bpb_quant (lower is better)", fontsize=12)
ax1.set_title(f"ChrisGoesGolfing: {len(df)} Experiments, {len(kept_v)} Kept", fontsize=14)
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.2)
margin = (baseline_bpb - best) * 0.15 if baseline_bpb > best else 0.005
ax1.set_ylim(best - margin, baseline_bpb + margin)

# Artifact size plot
valid_non_crash = valid[valid["artifact_bytes"] > 0]
ax2.bar(valid_non_crash.index, valid_non_crash["artifact_bytes"] / 1e6, color=["#2ecc71" if s == "KEEP" else "#cccccc" for s in valid_non_crash["status"]], alpha=0.7)
ax2.axhline(y=16.0, color="red", linestyle="--", linewidth=1.5, label="16MB limit")
ax2.set_xlabel("Experiment #", fontsize=12)
ax2.set_ylabel("Artifact Size (MB)", fontsize=12)
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
baseline_bpb = df.iloc[0]["val_bpb_quant"]
best_bpb = kept["val_bpb_quant"].min()
best_row = kept.loc[kept["val_bpb_quant"].idxmin()]

print(f"Baseline val_bpb_quant:  {baseline_bpb:.6f}")
print(f"Best val_bpb_quant:      {best_bpb:.6f}")
print(f"Total improvement:       {baseline_bpb - best_bpb:.6f} ({(baseline_bpb - best_bpb) / baseline_bpb * 100:.2f}%)")
print(f"Best experiment:         {best_row['description']}")
print(f"Best artifact size:      {best_row['artifact_bytes']:.0f} bytes ({best_row['artifact_bytes'] / 1e6:.2f} MB)")
print()

print("Cumulative improvements:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: bpb_q={row['val_bpb_quant']:.6f} artifact={row['artifact_bytes']:.0f}  {desc}")

## Top Hits (Ranked by Improvement)

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
kept["prev_bpb"] = kept["val_bpb_quant"].shift(1)
kept["delta"] = kept["prev_bpb"] - kept["val_bpb_quant"]
hits = kept.iloc[1:].copy()
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'BPB_Q':>10}  {'Artifact':>10}  Description")
print("-" * 90)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_bpb_quant']:.6f}  {row['artifact_bytes']:10.0f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  {'':>10}  TOTAL improvement over baseline")